In [1]:
# Import Required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Data Loading
df_train_FD001 = pd.read_csv("/home/jovyan/work/jet_engine_rul/Data/train_FD001.txt", sep=' ', header=None)
df_train_FD002 = pd.read_csv("/home/jovyan/work/jet_engine_rul/Data/train_FD002.txt", sep=' ', header=None)
df_train_FD003 = pd.read_csv("/home/jovyan/work/jet_engine_rul/Data/train_FD003.txt", sep=' ', header=None)
df_train_FD004 = pd.read_csv("/home/jovyan/work/jet_engine_rul/Data/train_FD004.txt", sep=' ', header=None)

# Remove null data (columns 26 and 27)
df_train_FD001 = df_train_FD001.iloc[:, :26]
df_train_FD002 = df_train_FD002.iloc[:, :26]
df_train_FD003 = df_train_FD003.iloc[:, :26]
df_train_FD004 = df_train_FD004.iloc[:, :26]

# Define column names
columns = ['Unit Number', 'Time (Cycles)', 'Altitude', 'Mach Number', 'TRA', 
           "T2", "T24", "T30", "T50", "P2", "P15", "P30", "Nf", "Nc", "epr", 
           "Ps30", "phi", "NRf", "NRc", "BPR", "farB", "htBleed", "Nf_dmd", 
           "PCNfR_dmd", "W31", "W32"]

df_train_FD001.columns = columns
df_train_FD002.columns = columns
df_train_FD003.columns = columns
df_train_FD004.columns = columns

# Uniquify Unit Numbers
df_train_FD002["Unit Number"] += df_train_FD001["Unit Number"].max()
df_train_FD003["Unit Number"] += df_train_FD002["Unit Number"].max()
df_train_FD004["Unit Number"] += df_train_FD003["Unit Number"].max()

# Merge datasets
df_train = pd.concat([df_train_FD001, df_train_FD002, df_train_FD003, df_train_FD004], axis=0)

# Cluster operational conditions
df_conditions = df_train[["Altitude", "Mach Number", "TRA"]]
scaler = StandardScaler()
df_conditions_scaled = scaler.fit_transform(df_conditions)

# Perform KMeans clustering for 6 operational conditions
km = KMeans(n_clusters=6, random_state=42)
df_train["Conditions"] = km.fit_predict(df_conditions_scaled)

# Calculate RUL for each unit
df_train["Fail Time"] = df_train.groupby("Unit Number")["Time (Cycles)"].transform('max')
df_train["RUL"] = df_train["Fail Time"] - df_train["Time (Cycles)"]

print("Data preparation complete!")
print(f"Dataset shape: {df_train.shape}")
print(f"Number of engines: {df_train['Unit Number'].nunique()}")
print(f"Number of operational conditions: {df_train['Conditions'].nunique()}")

Data preparation complete!
Dataset shape: (160359, 29)
Number of engines: 709
Number of operational conditions: 6


In [3]:
# Alternative 1: Expand historical window from 5 to 20 cycles
def prepare_enhanced_data(df, sensor_names, window_size=20):
    """
    Prepare data with historical window and additional features
    """
    features = []
    targets = []
    
    for engine_num in sorted(df['Unit Number'].unique()):
        engine_data = df[df['Unit Number'] == engine_num].sort_values('Time (Cycles)')
        
        # Use sliding window approach with larger window size
        for i in range(len(engine_data) - window_size):
            feature_row = []
            
            # Add temporal statistical features for each sensor in window
            window_data = engine_data.iloc[i:i+window_size]
            for sensor in sensor_names:
                if sensor in window_data.columns and not window_data[sensor].isna().all():
                    # Statistical features
                    feature_row.extend([
                        window_data[sensor].mean(),
                        window_data[sensor].std(),
                        window_data[sensor].min(),
                        window_data[sensor].max(),
                        window_data[sensor].median()
                    ])
            
            #  Add rate of change features (slope)
            for sensor in sensor_names:
                if sensor in window_data.columns and len(window_data) > 1:
                    # Calculate slope (rate of change) over the window
                    if window_data[sensor].iloc[-1] != window_data[sensor].iloc[0]:
                        # Slope = (final_value - initial_value) / (time_difference)
                        time_diff = (window_data['Time (Cycles)'].iloc[-1] - 
                                   window_data['Time (Cycles)'].iloc[0])
                        if time_diff != 0:
                            slope = (window_data[sensor].iloc[-1] - 
                                   window_data[sensor].iloc[0]) / time_diff
                        else:
                            slope = 0
                    else:
                        slope = 0
                    feature_row.append(slope)
            
            # Target: next RUL value
            target = engine_data.iloc[i + window_size]['RUL']
            
            features.append(feature_row)
            targets.append(target)
    
    return np.array(features), np.array(targets)

def train_linear_regression_with_tuning(df, sensor_names):
    """
    Train multiple models with hyperparameter tuning (Alternative 6)
    """
    
    # Prepare data with enhanced features
    X, y = prepare_enhanced_data(df, sensor_names, window_size=20)
    
    print(f"Enhanced data shape: {X.shape}")
    print(f"Number of features: {len(sensor_names) * (window_size + 5)}")  # window + stats + slope features
    
    unique_units = df_train[COLUMN_NAMES[Column.UnitNumber]].unique()
    train_units, test_units = train_test_split(unique_units, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    X_train = df_train[df_train[COLUMN_NAMES[Column.UnitNumber]].isin(train_units)].copy()
    X_test = df_train[df_train[COLUMN_NAMES[Column.UnitNumber]].isin(test_units)].copy()
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Store results
    results = {}
    
    # Train and evaluate each model
    linear_regression = LinearRegression()
    print(f"\n--- Training Linear Regression ---")
    
    # Fit model
    linear_regression.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_train_pred = linear_regression.predict(X_train_scaled)
    y_test_pred = linear_regression.predict(X_test_scaled)
    
    # Calculate metrics
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Store results
    results[name] = {
        'model': model,
        'scaler': scaler,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'y_test': y_test,
        'y_test_pred': y_test_pred
    }
    
    print(f"Train RMSE: {train_rmse:.4f}")
    print(f"Test RMSE: {test_rmse:.4f}")
    print(f"Train MAE: {train_mae:.4f}")
    print(f"Test MAE: {test_mae:.4f}")
    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    
    return results, X_test, y_test

# Identify sensor names (excluding non-feature columns)
not_features = ['Altitude', 'Mach Number', 'TRA']
weak_features = ['T2', 'P2', 'Nf_dmd', 'PCNfR_dmd']
irrelevant_features = not_features + weak_features
feature_names = df_train_FD001.columns
sensor_names = [name for name in feature_names if name not in irrelevant_features]

print("Sensor names identified:", sensor_names)
print("Total sensors:", len(sensor_names))

Sensor names identified: ['Unit Number', 'Time (Cycles)', 'T24', 'T30', 'T50', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'W31', 'W32']
Total sensors: 19


In [ ]:
# Train the enhanced models
results, X_test, y_test = train_linear_regression_with_tuning(df_train, sensor_names)

# Display comparison of all models
print("\n=== MODEL COMPARISON ===")
print("Enhanced Linear Regression Models with Better Features:")
print("-" * 60)

for model_name, metrics in results.items():
    print(f"\n{model_name}:")
    print(f"  Test RMSE: {metrics['test_rmse']:.4f}")
    print(f"  Test MAE: {metrics['test_mae']:.4f}")
    print(f"  Test R²: {metrics['test_r2']:.4f}")

# Select the best performing model
best_model_name = min(results.keys(), key=lambda x: results[x]['test_rmse'])
best_model = results[best_model_name]['model']
best_scaler = results[best_model_name]['scaler']

print(f"\nBest Model: {best_model_name}")
print(f"Best Test RMSE: {results[best_model_name]['test_rmse']:.4f}")

In [ ]:
def prepare_classification_data(df, sensor_names, window_size=20, threshold_percentile=20):
    """
    Convert RUL prediction to classification problem
    Alternative 7: Classification approach
    """
    features = []
    targets = []
    
    for engine_num in sorted(df['Unit Number'].unique()):
        engine_data = df[df['Unit Number'] == engine_num].sort_values('Time (Cycles)')
        
        # Calculate percentile threshold for classification
        # This creates categories: High RUL (normal), Medium RUL, Low RUL (critical)
        all_ruls = engine_data['RUL'].values
        threshold = np.percentile(all_ruls, threshold_percentile)  # Bottom 20% as critical
        
        for i in range(len(engine_data) - window_size):
            feature_row = []
            
            # Original sensor values from previous window_size cycles
            for j in range(window_size):
                row_idx = i + j
                for sensor in sensor_names:
                    if sensor in engine_data.columns:
                        feature_row.append(engine_data.iloc[row_idx][sensor])
            
            # Statistical features (Alternative 2)
            window_data = engine_data.iloc[i:i+window_size]
            for sensor in sensor_names:
                if sensor in window_data.columns and not window_data[sensor].isna().all():
                    feature_row.extend([
                        window_data[sensor].mean(),
                        window_data[sensor].std(),
                        window_data[sensor].min(),
                        window_data[sensor].max(),
                        window_data[sensor].median()
                    ])
            
            # Rate of change features (Alternative 2)
            for sensor in sensor_names:
                if sensor in window_data.columns and len(window_data) > 1:
                    if window_data[sensor].iloc[-1] != window_data[sensor].iloc[0]:
                        time_diff = (window_data['Time (Cycles)'].iloc[-1] - 
                                   window_data['Time (Cycles)'].iloc[0])
                        if time_diff != 0:
                            slope = (window_data[sensor].iloc[-1] - 
                                   window_data[sensor].iloc[0]) / time_diff
                        else:
                            slope = 0
                    else:
                        slope = 0
                    feature_row.append(slope)
            
            # Target: classify RUL into categories
            target_rul = engine_data.iloc[i + window_size]['RUL']
            
            # Create 3-class classification:
            # 0: High RUL (healthy) - above threshold
            # 1: Medium RUL (warning) - around threshold  
            # 2: Low RUL (critical) - below threshold
            if target_rul > threshold * 1.5:  # Healthy state
                target = 0
            elif target_rul > threshold * 0.5:  # Warning state
                target = 1
            else:  # Critical state
                target = 2
                
            features.append(feature_row)
            targets.append(target)
    
    return np.array(features), np.array(targets)

# Classification version training function
def train_classification_models(df, sensor_names):
    """
    Train classification models with enhanced features
    """
    # Prepare classification data
    X, y = prepare_classification_data(df, sensor_names, window_size=20)
    
    print(f"Classification data shape: {X.shape}")
    print(f"Number of classes: {len(np.unique(y))}")
    print(f"Class distribution: {np.bincount(y)}")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y  # Stratified sampling for balanced classes
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Classification models to try
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    }
    
    # Store results
    classification_results = {}
    
    # Train and evaluate each model
    for name, model in models.items():
        print(f"\n--- Training {name} ---")
        
        # Fit model
        model.fit(X_train_scaled, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train_scaled)
        y_test_pred = model.predict(X_test_scaled)
        
        # Calculate metrics
        train_accuracy = np.mean(y_train_pred == y_train)
        test_accuracy = np.mean(y_test_pred == y_test)
        
        # For detailed metrics, we'll use classification report
        from sklearn.metrics import classification_report, confusion_matrix
        
        print(f"Train Accuracy: {train_accuracy:.4f}")
        print(f"Test Accuracy: {test_accuracy:.4f}")
        
        # Store results
        classification_results[name] = {
            'model': model,
            'scaler': scaler,
            'train_accuracy': train_accuracy,
            'test_accuracy': test_accuracy,
            'y_test': y_test,
            'y_test_pred': y_test_pred
        }
        
        print(f"Classification Report for {name}:")
        print(classification_report(y_test, y_test_pred))
    
    return classification_results

# Train classification models
classification_results = train_classification_models(df_train, sensor_names)

# Display comparison of classification models
print("\n=== CLASSIFICATION MODEL COMPARISON ===")
print("Enhanced Classification Models with Better Features:")
print("-" * 60)

for model_name, metrics in classification_results.items():
    print(f"\n{model_name}:")
    print(f"  Test Accuracy: {metrics['test_accuracy']:.4f}")

In [ ]:
# Visualize results of best regression model
import matplotlib.pyplot as plt

# Get the best model results
best_regression_result = results[best_model_name]

# Plot actual vs predicted for the best model
plt.figure(figsize=(12, 5))

# Plot 1: Actual vs Predicted
plt.subplot(1, 2, 1)
plt.scatter(best_regression_result['y_test'], best_regression_result['y_test_pred'], alpha=0.6)
plt.plot([best_regression_result['y_test'].min(), best_regression_result['y_test'].max()], 
         [best_regression_result['y_test'].min(), best_regression_result['y_test'].max()], 'r--', lw=2)
plt.xlabel('Actual RUL')
plt.ylabel('Predicted RUL')
plt.title(f'Actual vs Predicted RUL\n{best_model_name}\nRMSE: {best_regression_result["test_rmse"]:.2f}')

# Plot 2: Residuals
plt.subplot(1, 2, 2)
residuals = best_regression_result['y_test'] - best_regression_result['y_test_pred']
plt.scatter(best_regression_result['y_test_pred'], residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted RUL')
plt.ylabel('Residuals')
plt.title('Residual Plot')

plt.tight_layout()
plt.show()

# Visualize classification results
plt.figure(figsize=(15, 5))

# Plot 1: Confusion matrix for best classification model
best_classification_model = max(classification_results.keys(), 
                               key=lambda x: classification_results[x]['test_accuracy'])

plt.subplot(1, 3, 1)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(classification_results[best_classification_model]['y_test'], 
                     classification_results[best_classification_model]['y_test_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_classification_model}')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Plot 2: Feature importance (if available)
plt.subplot(1, 3, 2)
if hasattr(best_model, 'coef_'):
    # For linear models, show coefficients
    feature_importance = pd.DataFrame({
        'Feature': [f'Feature_{i}' for i in range(len(best_model.coef_))],
        'Coefficient': best_model.coef_
    }).sort_values('Coefficient', key=abs, ascending=False)
    
    top_features = feature_importance.head(10)
    plt.barh(range(len(top_features)), top_features['Coefficient'])
    plt.yticks(range(len(top_features)), top_features['Feature'])
    plt.xlabel('Coefficient Value')
    plt.title('Top Feature Coefficients')
elif hasattr(best_model, 'feature_importances_'):
    # For tree-based models, show feature importances
    feature_importance = pd.DataFrame({
        'Feature': [f'Feature_{i}' for i in range(len(best_model.feature_importances_))],
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    top_features = feature_importance.head(10)
    plt.barh(range(len(top_features)), top_features['Importance'])
    plt.yticks(range(len(top_features)), top_features['Feature'])
    plt.xlabel('Importance')
    plt.title('Top Feature Importances')

# Plot 3: Model comparison
plt.subplot(1, 3, 3)
regression_metrics = ['RMSE', 'MAE', 'R²']
regression_values = [
    best_regression_result['test_rmse'],
    best_regression_result['test_mae'],
    best_regression_result['test_r2']
]

classification_metrics = ['Accuracy']
classification_values = [classification_results[best_classification_model]['test_accuracy']]

# Combine both types of metrics
all_metrics = regression_metrics + classification_metrics
all_values = regression_values + classification_values

bars = plt.bar(range(len(all_metrics)), all_values)
plt.xticks(range(len(all_metrics)), all_metrics)
plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.ylim(0, max(all_values) * 1.1)

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, all_values)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()